In [1]:
# Import required libraries
import mlflow
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "../../run")
from const import REPO_PATH
from experiment_config import TRAINGING_CONFIG

sys.path.insert(1, f"{REPO_PATH}")
from src.model.experiment_utils import *
from src.model.model_utils import *
from src.model.models import LSTMClassifier, TCNClassifier
from src.model.experiment_utils import run_multiclass_distribution_experiment_seq



/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
features_path = f"{TRAINGING_CONFIG['features_path']}/all_combined_features_2017-24.csv"
output_dir = f"{REPO_PATH}/models/ml_models"

In [3]:
seasons=sorted(TRAINGING_CONFIG['seasons'])
target_dfs=[pd.read_csv(f"{TRAINGING_CONFIG['processed_data_path']}/{season}/all_target_df.csv") for season in seasons[1:]]
for df in target_dfs:
    df['date']= pd.to_datetime(df['date'])
target_df = pd.concat(target_dfs, ignore_index=True)
target_columns= list(TRAINGING_CONFIG['target_ranges'].keys())

In [4]:
feature_df = pd.read_csv(features_path)
feature_df['date']= pd.to_datetime(feature_df['date'])
feature_df, target_df=align_on_keys(feature_df, target_df, TRAINGING_CONFIG['key_columns'])

In [ ]:
target_names=list(TRAINGING_CONFIG['TARGET_RANGES'].keys())
model_names=[]

In [ ]:
# Loop through each target and its associated models
for target in target_names:
    print(f"\nTarget: {target}")
    for model_name in model_names[target]:
        print(f"  Model: {model_name}")
        # Load model from MLflow
        model_uri = f"models:/{model_name}/latest"
        model = mlflow.pyfunc.load_model(model_uri)
        
        # Prepare data for SHAP (exclude key columns)
        X = feature_df.drop(columns=key_cols)
        
        # Use TreeExplainer for tree-based models, KernelExplainer otherwise
        try:
            explainer = shap.Explainer(model.predict, X)
        except Exception as e:
            print(f"    SHAP Explainer error: {e}")
            continue
        
        # Compute SHAP values
        shap_values = explainer(X)
        
        # Plot SHAP summary
        shap.summary_plot(shap_values, X, show=False)
        plt.title(f"SHAP Summary: {model_name} ({target})")
        plt.show()
